## SpectralWaste

In [1]:
import torch
from torchvision import transforms
import os
import glob
import yaml
import numpy as np
import tifffile
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import multiprocessing
from functools import partial
from sklearn.decomposition import PCA
from scipy.signal import savgol_filter

def collect_image_filesets(rgb_folder_path, hsi_folder_path):
    """
    Scans RGB and HSI folders to find matching file pairs.
    Assumes RGB files are .png and HSI files are .tiff, with matching basenames.
    """
    print(f"Collecting file pairs from RGB: {rgb_folder_path} and HSI: {hsi_folder_path}")
    
    # Find all RGB images (assuming .png, but could be .jpg)
    rgb_files = sorted(glob.glob(os.path.join(rgb_folder_path, "*.png")))
    if not rgb_files:
        rgb_files = sorted(glob.glob(os.path.join(rgb_folder_path, "*.jpg")))

    if not rgb_files:
        print(f"Warning: No RGB files (.png or .jpg) found in {rgb_folder_path}")
        return []

    image_sets = []
    for rgb_path in rgb_files:
        rgb_filename = os.path.basename(rgb_path)
        file_stem = Path(rgb_filename).stem
        
        # Construct the expected HSI filename
        hsi_filename = f"{file_stem}.tiff"
        hsi_path = os.path.join(hsi_folder_path, hsi_filename)
        
        # Check if the corresponding HSI file exists
        if os.path.exists(hsi_path):
            image_sets.append({
                'rgb_file': rgb_filename,
                'hsi_file': hsi_filename,
            })
        else:
            print(f"  - Warning: No matching HSI file found for {rgb_filename} (expected {hsi_filename})")
            
    print(f"Found {len(image_sets)} matching RGB/HSI file pairs.")
    return image_sets


def calculate_hsi_statistics_from_tiffs(tiff_folder_path, spectral_channels=224):
    """
    Calculates HSI normalization statistics (mean, std) directly from raw .tiff files.
    """
    channel_sums = np.zeros(spectral_channels, dtype=np.float64)
    channel_sq_sums = np.zeros(spectral_channels, dtype=np.float64)
    pixel_counts = np.zeros(spectral_channels, dtype=np.int64)
    
    tiff_files = sorted(list(Path(tiff_folder_path).glob('*.tiff')))
    
    if not tiff_files:
        print(f"Error: No .tiff files found in {tiff_folder_path}")
        return None

    print(f"Processing {len(tiff_files)} TIFF files for normalization statistics...")
    
    for file_path in tqdm(tiff_files, desc="Calculating HSI stats from TIFFs"):
        try:
            hsi_data = tifffile.imread(file_path)

            # Ensure data is (C, H, W)
            if hsi_data.ndim == 3 and hsi_data.shape[2] == spectral_channels:
                hsi_data = hsi_data.transpose(2, 0, 1)

            if hsi_data.shape[0] != spectral_channels:
                print(f"Warning: Expected {spectral_channels} channels, got {hsi_data.shape[0]} in {file_path.name}")
                continue

            # Cast to float64 before any arithmetic to avoid integer overflow
            hsi_data = hsi_data.astype(np.float64)

            # Boolean mask of valid (non-background) pixels
            non_zero_mask = (hsi_data != 0)  # shape (C, H, W)

            # Accumulate sums in a way that avoids using `where` (more portable)
            channel_sums += np.sum(hsi_data * non_zero_mask, axis=(1, 2))
            channel_sq_sums += np.sum((hsi_data * non_zero_mask) ** 2, axis=(1, 2))
            pixel_counts += np.sum(non_zero_mask, axis=(1, 2))
                        
        except Exception as e:
            print(f"Error processing {file_path.name}: {e}")
            continue
    
    # Calculate final statistics safely
    # means and mean_of_squares only where pixel_counts != 0
    channel_means = np.divide(channel_sums, pixel_counts, out=np.zeros_like(channel_sums), where=pixel_counts!=0)
    mean_of_squares = np.divide(channel_sq_sums, pixel_counts, out=np.zeros_like(channel_sq_sums), where=pixel_counts!=0)

    # variance = E[x^2] - (E[x])^2 ; clip tiny negative values to 0
    channel_vars = mean_of_squares - np.square(channel_means)
    channel_vars = np.clip(channel_vars, 0.0, None)

    channel_stds = np.sqrt(channel_vars, out=np.zeros_like(channel_vars))
    
    print("HSI Statistics calculation complete.")
    return {'means': channel_means, 'stds': channel_stds}



def process_single_image_to_npz(file_set, 
                                split_name_arg,
                                src_rgb_folder_arg, src_hsi_folder_arg,
                                dest_npz_folder_base_arg, 
                                num_expected_hsi_channels_arg,
                                hsi_transform, rgb_transform):
    """
    Processes a single RGB/HSI pair, applies transforms, and saves them as named arrays in a single .npz file.
    """
    rgb_filename = file_set['rgb_file']
    hsi_filename = file_set['hsi_file']
    
    src_rgb_path = os.path.join(src_rgb_folder_arg, rgb_filename)
    src_hsi_path = os.path.join(src_hsi_folder_arg, hsi_filename)
    
    dest_split_folder = os.path.join(dest_npz_folder_base_arg, split_name_arg)
    # Ensure output folder exists
    os.makedirs(dest_split_folder, exist_ok=True)

    # Build a stable output stem and npz filepath
    output_stem = Path(rgb_filename).stem
    npz_filepath = os.path.join(dest_split_folder, f"{output_stem}.npz")

    try:
        # --- Load Raw Data ---
        if not os.path.exists(src_rgb_path):
            return f"Warning: RGB file not found {src_rgb_path}, skipping."
        rgb_pil = Image.open(src_rgb_path).convert('RGB')

        if not os.path.exists(src_hsi_path):
            return f"Warning: HSI file not found {src_hsi_path}, skipping."
        hsi_data_raw = tifffile.imread(src_hsi_path)
        
        # --- Ensure HSI is C,H,W ---
        hsi_chw = None
        if hsi_data_raw.ndim == 3:
            if hsi_data_raw.shape[0] == num_expected_hsi_channels_arg:
                hsi_chw = hsi_data_raw
            elif hsi_data_raw.shape[2] == num_expected_hsi_channels_arg:
                hsi_chw = hsi_data_raw.transpose(2, 0, 1)
        
        if hsi_chw is None:
            return f"Warning: HSI data in {hsi_filename} has unexpected shape."

        # --- Apply Transforms ---
        rgb_tensor_transformed = rgb_transform(rgb_pil)
        hsi_tensor_raw = torch.from_numpy(hsi_chw.astype(np.float32))
        hsi_tensor_transformed = hsi_transform(hsi_tensor_raw)

        # --- Stack and Save ---
        # 🔥 CHANGE: DO NOT STACK. SAVE AS SEPARATE NAMED ARRAYS.
        rgb_numpy = rgb_tensor_transformed.numpy()
        hsi_numpy = hsi_tensor_transformed.numpy()
        
        # Use np.savez_compressed to save multiple arrays into one file
        np.savez_compressed(npz_filepath, rgb=rgb_numpy, hsi=hsi_numpy)
        
        return None
    except Exception as e:
        return f"Error processing {output_stem}: {e}"


def execute_npz_processing_for_split(image_file_sets, split_name_main,
                                     src_rgb_folder_main, src_hsi_folder_main,
                                     dest_npz_folder_base_main,
                                     num_expected_hsi_channels_main,
                                     hsi_transform, rgb_transform,
                                     num_processes=10):
    if not image_file_sets:
        print(f"No image file sets for {split_name_main}. Skipping.")
        return

    dest_split_folder_main = os.path.join(dest_npz_folder_base_main, split_name_main)
    os.makedirs(dest_split_folder_main, exist_ok=True)
    print(f"Processing {split_name_main}. Saving pre-processed .npz files to: {dest_split_folder_main} using {num_processes} processes.")

    worker_partial = partial(process_single_image_to_npz,
                             split_name_arg=split_name_main,
                             src_rgb_folder_arg=src_rgb_folder_main,
                             src_hsi_folder_arg=src_hsi_folder_main,
                             dest_npz_folder_base_arg=dest_npz_folder_base_main,
                             num_expected_hsi_channels_arg=num_expected_hsi_channels_main,
                             hsi_transform=hsi_transform,
                             rgb_transform=rgb_transform)
    
    errors = []
    processed_count = 0

    # 🔥 FIX: Add a single-process mode for debugging and compatibility with Jupyter on Windows.
    if num_processes > 1:
        # Use multiprocessing
        with multiprocessing.Pool(processes=num_processes) as pool:
            results = list(tqdm(pool.imap_unordered(worker_partial, image_file_sets), total=len(image_file_sets), desc=f"Saving {split_name_main} (.npz)"))
            for res in results:
                if res is None:
                    processed_count += 1
                else:
                    errors.append(res)
    else:
        # Use a simple loop for single-process execution
        print("Running in single-process mode.")
        for file_set in tqdm(image_file_sets, desc=f"Saving {split_name_main} (.npz)"):
            res = worker_partial(file_set)
            if res is None:
                processed_count += 1
            else:
                errors.append(res)

    if errors:
        print(f"Finished .npz processing for {processed_count} files in {split_name_main} with {len(errors)} errors.")
        # Optionally print some errors
        for err in errors[:5]:
            print(f"  - {err}")
    else:
        print(f"Finished .npz processing for {processed_count} files in {split_name_main}.")


class MaskedNormalize(torch.nn.Module):
    def __init__(self, mean, std, zero_std_epsilon=1.0e-6):
        super().__init__()
        # Store as 1-D tensors (do not reshape here; reshape at forward for device)
        self.register_buffer('_mean_1d', torch.as_tensor(mean).detach().clone().float().view(-1))
        self.register_buffer('_std_1d',  torch.as_tensor(std).detach().clone().float().view(-1))
        # Replace zero stds with a small epsilon to avoid division by zero
        self._std_1d[self._std_1d == 0] = float(zero_std_epsilon)

    def forward(self, tensor: torch.Tensor) -> torch.Tensor:
        # tensor expected shape (C, H, W)
        if tensor is None:
            return tensor
        C = tensor.shape[0]
        # Ensure stats length matches channels
        if self._mean_1d.numel() != C or self._std_1d.numel() != C:
            raise ValueError(f"MaskedNormalize: stats length ({self._mean_1d.numel()}/{self._std_1d.numel()}) "
                             f"does not match tensor channels ({C}).")

        # Move and reshape stats to (C,1,1) on the same device/dtype
        mean = self._mean_1d.to(tensor.device, tensor.dtype).view(C, 1, 1)
        std =  self._std_1d.to(tensor.device, tensor.dtype).view(C, 1, 1)

        # Element-wise mask per channel so we preserve zeros per-channel
        non_zero_mask = (tensor != 0)  # shape (C,H,W)

        # Compute normalized values for all pixels
        normalized_all = (tensor - mean) / std

        # Start with zeros and copy only where original had non-zero values
        out = torch.zeros_like(tensor)
        out[non_zero_mask] = normalized_all[non_zero_mask]
        return out

In [2]:
# --- 1. DEFINE PATHS ---
# Original source data
src_base = Path(r"C:\Users\jon86439\Desktop\Master_Thesis\Experiment_MA_Publication\data\SpectralWaste")
# Destination for the new, fast .npz dataset
dest_npz_base = Path(r"C:\Users\jon86439\Desktop\Master_Thesis\Experiment_MA_Publication\data\SpectralWaste\labelled_npz")
# Path to save/load your dataset's statistics YAML file
stats_file_path = Path(r"C:\Users\jon86439\Desktop\BCAF\training_scripts\configs\SpectralWaste\data\hsi_stats.yaml")
# Constants
rgb_resize_size = 256
num_hsi_channels = 224

hsi_stats_calc_path = src_base / 'hyper' / 'train'

In [ ]:
# --- 2. CALCULATE AND SAVE HSI NORMALIZATION STATS ---
print("--- Step 1: Calculating HSI Normalization Statistics ---")
# We calculate stats ONLY from the 'train' split to avoid data leakage

hsi_stats = calculate_hsi_statistics_from_tiffs(hsi_stats_calc_path, spectral_channels=num_hsi_channels)

if hsi_stats:
    stats_file_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with open(stats_file_path, 'w') as f:
            yaml.dump({
                'standardize': {
                    'hsi_global_means': hsi_stats['means'].tolist(),
                    'hsi_global_stds': hsi_stats['stds'].tolist()
                }
            }, f, indent=4)
        print(f"✅ HSI normalization statistics saved to: {stats_file_path}")
    except Exception as e:
        print(f"Error saving stats file: {e}")

     # --- 3. DEFINE TRANSFORMS ---
    print("\n--- Step 2: Defining Transformation Pipelines ---")
    hsi_mean = torch.tensor(hsi_stats['means'], dtype=torch.float32)
    hsi_std = torch.tensor(hsi_stats['stds'], dtype=torch.float32)

    # HSI transform no longer resizes, it only normalizes.
    hsi_transform = transforms.Compose([])
    # RGB transform uses the new dedicated resize variable.
    rgb_transform = transforms.Compose([
        transforms.Resize((rgb_resize_size, rgb_resize_size), antialias=True),
        transforms.ToTensor(),
    ])
    print("Image transforms for HSI (no resize) and RGB are ready.")


    # --- 4. EXECUTE PRE-PROCESSING FOR ALL SPLITS ---
    print("\n--- Step 3: Processing images to NumPy format ---")
    for split in ['train', 'val', 'test']:
        print(f"\n--- Processing split: {split} ---")
        
        # Define separate paths for RGB and HSI source folders
        src_rgb_folder = src_base / 'rgb' / split
        src_hsi_folder = src_base / 'hyper' / split

        if not src_rgb_folder.exists():
            print(f"Source RGB folder not found, skipping: {src_rgb_folder}")
            continue
        if not src_hsi_folder.exists():
            print(f"Source HSI folder not found, skipping: {src_hsi_folder}")
            continue

        # Pass the correct, separate folder paths to the collection function
        image_sets = collect_image_filesets(str(src_rgb_folder), str(src_hsi_folder))

        if image_sets:
            execute_npz_processing_for_split(
                image_file_sets=image_sets, 
                split_name_main=split,
                # Pass the correct, separate folder paths to the processing function
                src_rgb_folder_main=str(src_rgb_folder), 
                src_hsi_folder_main=str(src_hsi_folder), 
                dest_npz_folder_base_main=str(dest_npz_base / 'images'),
                num_expected_hsi_channels_main=num_hsi_channels,
                hsi_transform=hsi_transform,
                rgb_transform=rgb_transform,
                num_processes=1
            )

    print(f"\n\n✅✅✅ All processing complete! Your final dataset is at: '{dest_npz_base}' ✅✅✅")
else:
    print("❌ FATAL ERROR: Could not calculate HSI statistics. Aborting pre-processing.")

pca - ablation

In [3]:
def calculate_pca_from_npz(npz_train_folder_path, pca_configs, num_hsi_channels_to_expect):
    """
    Calculates PCA transformations from pre-processed HSI data stored in .npz files.
    Assumes .npz files contain stacked (RGB+HSI) data and HSI is already normalized.
    """
    print(f"--- Starting PCA Calculation from Pre-processed .npz Files ---")
    print(f"Source folder: {npz_train_folder_path}")

    npz_files = sorted(glob.glob(os.path.join(npz_train_folder_path, "*.npz")))
    if not npz_files:
        print("Error: No .npz files found for PCA calculation. Aborting.")
        return

    all_pixel_data_list = []

    for file_path in tqdm(npz_files, desc="Loading HSI data from .npz for PCA"):
        try:
            #
            data = np.load(file_path)
            hsi_data_unnormalized = data['hsi']
            
            if hsi_data_unnormalized.shape[0] != num_hsi_channels_to_expect:
                print(f"Warning: Channel mismatch in {file_path}. Expected {num_hsi_channels_to_expect}, got {hsi_data_normalized.shape[0]}. Skipping.")
                continue

            mean_np = hsi_stats['means'].reshape(-1, 1, 1)
            std_np = hsi_stats['stds'].reshape(-1, 1, 1)
            std_np[std_np == 0] = 1.0e-6 # Avoid division by zero

            # Create mask and apply normalization
            non_zero_mask = (hsi_data_unnormalized != 0)
            hsi_data_normalized = np.zeros_like(hsi_data_unnormalized, dtype=np.float32)
            
            # --- SIMPLIFIED NORMALIZATION ---
            # Use broadcasting to apply normalization efficiently
            normalized_values = (hsi_data_unnormalized - mean_np) / std_np
            hsi_data_normalized[non_zero_mask] = normalized_values[non_zero_mask]
            # Create a mask of valid spectra (any non-zero value across channels)
            # This works because our manual normalization preserves background zeros.
            valid_mask = (hsi_data_normalized != 0).any(axis=0)
            
            # Extract only valid spectra, shape (C, N_valid_pixels)
            spectra_valid = hsi_data_normalized[:, valid_mask]
            if spectra_valid.size == 0:
                continue

            # Transpose to (N_valid_pixels, C) for PCA
            all_pixel_data_list.append(spectra_valid.T)
        except Exception as e:
            print(f"Error processing file {file_path} for PCA: {e}")

    if not all_pixel_data_list:
        print("Error: No pixel data collected for PCA. Aborting.")
        return

    full_pixel_dataset = np.vstack(all_pixel_data_list)
    print(f"Total pixel dataset shape for PCA: {full_pixel_dataset.shape}")

    # --- Fit PCA models ---
    for config in pca_configs:
        n_components = config['n_components']
        output_path = config['output_path']
        print(f"\nFitting PCA with {n_components} components...")
        
        pca_model = PCA(n_components=n_components, random_state=42)
        pca_model.fit(full_pixel_dataset)

        pca_params = {
            'pca_parameters': {
                'mean': pca_model.mean_.tolist(),
                'components': pca_model.components_.tolist(),
            },
            'hsi_normalization_method_used_on_input': 'standardize (pre-applied)'
        }
        
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w') as f:
            yaml.dump(pca_params, f, indent=4)
        print(f"Saved PCA transformation ({n_components} components) to {output_path}")

def calculate_savgol_pca_from_npz(npz_train_folder_path, pca_configs, savgol_params, num_hsi_channels_to_expect, hsi_stats, num_processes=10):
    """
    Applies Savitzky-Golay filter and then calculates PCA from pre-processed .npz files.
    """
    print(f"--- Starting Savitzky-Golay + PCA from Pre-processed .npz Files ---")
    print(f"Source folder: {npz_train_folder_path}")
    print(f"Savitzky-Golay params: {savgol_params}")

    npz_files = sorted(glob.glob(os.path.join(npz_train_folder_path, "*.npz")))
    if not npz_files:
        print("Error: No .npz files found. Aborting.")
        return

    # --- Apply SavGol Filter in Parallel ---
    worker_partial = partial(_worker_savgol_npz,
                             savgol_window=savgol_params['window'],
                             savgol_polyorder=savgol_params['polyorder'],
                             savgol_deriv=savgol_params['deriv'],
                             num_hsi_channels_ref=num_hsi_channels_to_expect,
                             hsi_stats=hsi_stats)

    results = []
    if num_processes > 1:
        with multiprocessing.Pool(processes=num_processes) as pool:
            results = list(tqdm(pool.imap(worker_partial, npz_files), total=len(npz_files), desc="Applying SavGol Filter"))
    else:
        print("Running SavGol in single-process mode.")
        for file_path in tqdm(npz_files, desc="Applying SavGol Filter"):
            results.append(worker_partial(file_path))

    savgol_filtered_data = [res for res in results if res is not None]

    if not savgol_filtered_data:
        print("Error: No data collected after Savitzky-Golay filtering. Aborting.")
        return
    
    full_pixel_dataset = np.vstack(savgol_filtered_data)
    print(f"Total SavGol-filtered pixel dataset shape for PCA: {full_pixel_dataset.shape}")

    # --- Fit PCA models on SavGol-filtered data ---
    for config in pca_configs:
        n_components = config['n_components']
        output_path = config['output_path']
        print(f"\nFitting PCA with {n_components} components on SavGol data...")
        
        pca_model = PCA(n_components=n_components, random_state=42)
        pca_model.fit(full_pixel_dataset)

        pca_params = {
            'pca_parameters': {
                'mean': pca_model.mean_.tolist(),
                'components': pca_model.components_.tolist(),
            },
            'preprocessing_details': {
                'input_data_normalization_before_savgol': 'standardize (pre-applied)',
                'savitzky_golay_filter': savgol_params
            }
        }
        
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w') as f:
            yaml.dump(pca_params, f, indent=4)
        print(f"Saved SavGol+PCA transformation ({n_components} components) to {output_path}")

def calculate_pca_from_npz(npz_train_folder_path, pca_configs, num_hsi_channels_to_expect, hsi_stats):
    """
    Calculates PCA transformations from pre-processed HSI data stored in .npz files.
    Assumes .npz files contain stacked (RGB+HSI) data and HSI is already normalized.
    """
    print(f"--- Starting PCA Calculation from Pre-processed .npz Files ---")
    print(f"Source folder: {npz_train_folder_path}")

    npz_files = sorted(glob.glob(os.path.join(npz_train_folder_path, "*.npz")))
    if not npz_files:
        print("Error: No .npz files found for PCA calculation. Aborting.")
        return

    all_pixel_data_list = []

    for file_path in tqdm(npz_files, desc="Loading HSI data from .npz for PCA"):
        try:
            #
            data = np.load(file_path)
            hsi_data_unnormalized = data['hsi']
            
            if hsi_data_unnormalized.shape[0] != num_hsi_channels_to_expect:
                print(f"Warning: Channel mismatch in {file_path}. Expected {num_hsi_channels_to_expect}, got {hsi_data_unnormalized.shape[0]}. Skipping.")
                continue

            mean_np = hsi_stats['means'].reshape(-1, 1, 1)
            std_np = hsi_stats['stds'].reshape(-1, 1, 1)
            std_np[std_np == 0] = 1.0e-6 # Avoid division by zero

            # Create mask and apply normalization
            non_zero_mask = (hsi_data_unnormalized != 0)
            hsi_data_normalized = np.zeros_like(hsi_data_unnormalized, dtype=np.float32)
            
            # --- SIMPLIFIED NORMALIZATION ---
            # Use broadcasting to apply normalization efficiently
            normalized_values = (hsi_data_unnormalized - mean_np) / std_np
            hsi_data_normalized[non_zero_mask] = normalized_values[non_zero_mask]
            # Create a mask of valid spectra (any non-zero value across channels)
            # This works because our manual normalization preserves background zeros.
            valid_mask = (hsi_data_normalized != 0).any(axis=0)
            
            # Extract only valid spectra, shape (C, N_valid_pixels)
            spectra_valid = hsi_data_normalized[:, valid_mask]
            if spectra_valid.size == 0:
                continue

            # Transpose to (N_valid_pixels, C) for PCA
            all_pixel_data_list.append(spectra_valid.T)
        except Exception as e:
            print(f"Error processing file {file_path} for PCA: {e}")

    if not all_pixel_data_list:
        print("Error: No pixel data collected for PCA. Aborting.")
        return

    full_pixel_dataset = np.vstack(all_pixel_data_list)
    print(f"Total pixel dataset shape for PCA: {full_pixel_dataset.shape}")

    # --- Fit PCA models ---
    for config in pca_configs:
        n_components = config['n_components']
        output_path = config['output_path']
        print(f"\nFitting PCA with {n_components} components...")
        
        pca_model = PCA(n_components=n_components, random_state=42)
        pca_model.fit(full_pixel_dataset)

        pca_params = {
            'pca_parameters': {
                'mean': pca_model.mean_.tolist(),
                'components': pca_model.components_.tolist(),
            },
            'hsi_normalization_method_used_on_input': 'standardize (pre-applied)'
        }
        
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w') as f:
            yaml.dump(pca_params, f, indent=4)
        print(f"Saved PCA transformation ({n_components} components) to {output_path}")

In [4]:
# --- 5. CALCULATE PCA AND SAVGOL-PCA FROM PRE-PROCESSED .NPZ FILES ---
print("\n--- Step 4: Calculating PCA and SavGol+PCA from .npz files ---")



# Define output paths for the new PCA files
base_config_path = Path("/home/jon86439/BCAF/training_scripts/configs/SpectralWaste")
hsi_stats_yaml = Path("/home/jon86439/BCAF/training_scripts/configs/SpectralWaste/data/hsi_stats.yaml")
pca_output_path_5 = base_config_path / "SpectralWaste_pca5_from_npz.yaml"
savgol_pca_output_path_5 = base_config_path / "SpectralWaste_savgol_pca5_from_npz.yaml"


# Path to the newly created .npz training data
npz_train_path = Path("/home/jon86439/Experiment_MA_Publication/data/SpectralWaste/labelled_npz/images/train")

# Load YAML stats and convert to numpy arrays expected by the PCA functions
if not hsi_stats_yaml.exists():
    raise FileNotFoundError(f"HSI stats YAML not found: {hsi_stats_yaml}. Run stats calc cell first.")
stats_yaml = yaml.safe_load(hsi_stats_yaml.read_text())
hsi_stats = {
    'means': np.array(stats_yaml['standardize']['hsi_global_means'], dtype=np.float32),
    'stds':  np.array(stats_yaml['standardize']['hsi_global_stds'], dtype=np.float32),
}

# --- Run Standard PCA ---
pca_configs_to_run = [
    {'n_components': 5, 'output_path': str(pca_output_path_5)},
]









def _worker_savgol_npz(file_path, savgol_window, savgol_polyorder, savgol_deriv, 
                       num_hsi_channels_ref, hsi_stats):
    """
    Worker function to load, normalize, apply Savitzky-Golay filter, and return valid spectra.
    """
    try:
        data = np.load(file_path)
        hsi_data_unnormalized = data['hsi']
        
        if hsi_data_unnormalized.shape[0] != num_hsi_channels_ref:
            return None

        # Normalize
        mean_np = hsi_stats['means'].reshape(-1, 1, 1)
        std_np = hsi_stats['stds'].reshape(-1, 1, 1)
        std_np[std_np == 0] = 1.0e-6

        non_zero_mask = (hsi_data_unnormalized != 0)
        hsi_data_normalized = np.zeros_like(hsi_data_unnormalized, dtype=np.float32)
        normalized_values = (hsi_data_unnormalized - mean_np) / std_np
        hsi_data_normalized[non_zero_mask] = normalized_values[non_zero_mask]

        # Apply Savitzky-Golay filter channel-wise
        C, H, W = hsi_data_normalized.shape
        hsi_filtered = np.zeros_like(hsi_data_normalized)
        
        for h in range(H):
            for w in range(W):
                spectrum = hsi_data_normalized[:, h, w]
                if np.any(spectrum != 0):
                    try:
                        filtered_spectrum = savgol_filter(
                            spectrum, 
                            window_length=savgol_window,
                            polyorder=savgol_polyorder,
                            deriv=savgol_deriv
                        )
                        hsi_filtered[:, h, w] = filtered_spectrum
                    except:
                        hsi_filtered[:, h, w] = spectrum

        # Extract valid spectra
        valid_mask = (hsi_filtered != 0).any(axis=0)
        spectra_valid = hsi_filtered[:, valid_mask]
        
        if spectra_valid.size == 0:
            return None
            
        return spectra_valid.T  # (N_valid_pixels, C)
        
    except Exception as e:
        print(f"Error in worker for {file_path}: {e}")
        return None
    
    
calculate_pca_from_npz(
    npz_train_folder_path=str(npz_train_path),
    pca_configs=pca_configs_to_run,
    num_hsi_channels_to_expect=num_hsi_channels,
    hsi_stats=hsi_stats
)

#--- Run Savitzky-Golay + PCA ---
savgol_pca_configs_to_run = [
    {'n_components': 5, 'output_path': str(savgol_pca_output_path_5)},
]

savgol_parameters = {
    'window': 7,
    'polyorder': 1,
    'deriv': 1
}
calculate_savgol_pca_from_npz(
    hsi_stats=hsi_stats,
    npz_train_folder_path=str(npz_train_path),
    pca_configs=savgol_pca_configs_to_run,
    savgol_params=savgol_parameters,
    num_hsi_channels_to_expect=num_hsi_channels,
    num_processes=1
)

print("\n\n✅✅✅ PCA and SavGol+PCA calculations from .npz files are complete! ✅✅✅")


--- Step 4: Calculating PCA and SavGol+PCA from .npz files ---
--- Starting PCA Calculation from Pre-processed .npz Files ---
Source folder: /home/jon86439/Experiment_MA_Publication/data/SpectralWaste/labelled_npz/images/train


Loading HSI data from .npz for PCA: 100%|██████████| 514/514 [02:57<00:00,  2.89it/s]


Total pixel dataset shape for PCA: (33685504, 224)

Fitting PCA with 5 components...
Saved PCA transformation (5 components) to /home/jon86439/BCAF/training_scripts/configs/SpectralWaste/SpectralWaste_pca5_from_npz.yaml
--- Starting Savitzky-Golay + PCA from Pre-processed .npz Files ---
Source folder: /home/jon86439/Experiment_MA_Publication/data/SpectralWaste/labelled_npz/images/train
Savitzky-Golay params: {'window': 7, 'polyorder': 1, 'deriv': 1}
Running SavGol in single-process mode.


Applying SavGol Filter: 100%|██████████| 514/514 [1:11:03<00:00,  8.29s/it]


Total SavGol-filtered pixel dataset shape for PCA: (33685504, 224)

Fitting PCA with 5 components on SavGol data...
Saved SavGol+PCA transformation (5 components) to /home/jon86439/BCAF/training_scripts/configs/SpectralWaste/SpectralWaste_savgol_pca5_from_npz.yaml


✅✅✅ PCA and SavGol+PCA calculations from .npz files are complete! ✅✅✅


# unlabelled

In [ ]:
import random
from math import ceil

# Configure: adjust if your unlabeled folders are named differently
src_base = Path(r"C:\Users\jon86439\Desktop\spectralwaste_segmentation_unlabeled\spectralwaste_segmentation_unlabeled")
dest_npz_base = Path(r"C:\Users\jon86439\Desktop\Master-Thesis-GIT\Experiment_MA_Publication\data\SpectralWaste\unlabelled_npz")
stats_file_path = Path(r"C:\Users\jon86439\Desktop\Master-Thesis-GIT\Experiment_MA_Publication\configs\data\SpectralWaste\SpectralWaste_hsi_stats_npz.yaml")

rgb_resize_size = 256
num_hsi_channels = 224

unlabeled_hsi_folder = src_base / 'hyper' / 'unlabeled'  # change 'unlabelled' if needed
unlabeled_rgb_folder = src_base / 'rgb' / 'unlabeled'     # may not exist; zeros will be used
dest_images_base = dest_npz_base / 'images'                # same output layout as other steps
val_fraction = 0.10
random_seed = 42

# Load stats (must exist from earlier cell)
if not stats_file_path.exists():
    print(f"Stats file not found: {stats_file_path}. Run stats calculation first.")
else:
    stats = yaml.safe_load(stats_file_path.read_text())
    means = torch.tensor(stats['standardize']['hsi_global_means'], dtype=torch.float32)
    stds  = torch.tensor(stats['standardize']['hsi_global_stds'], dtype=torch.float32)

    # transforms (make unlabeled use same logic as labelled pipeline)
    # LABELLED used: hsi_transform = transforms.Compose([])  (no normalization here)
    hsi_transform = transforms.Compose([])

    rgb_transform = transforms.Compose([
        transforms.Resize((rgb_resize_size, rgb_resize_size), antialias=True),
        transforms.ToTensor(),
    ])
    # collect HSI files
    if not unlabeled_hsi_folder.exists():
        print(f"Unlabeled HSI folder not found: {unlabeled_hsi_folder}")
    else:
        hsi_files = sorted(list(unlabeled_hsi_folder.glob("*.tiff")))
        if not hsi_files:
            print(f"No .tiff files found in {unlabeled_hsi_folder}")
        else:
            # deterministic shuffle + split
            random.seed(random_seed)
            files = [p for p in hsi_files]
            random.shuffle(files)
            n = len(files)
            n_val = max(1, int(ceil(n * val_fraction))) if n>1 else 1
            val_files = files[:n_val]
            train_files = files[n_val:]

            def _process_list(file_list, split_name):
                out_folder = dest_images_base / split_name
                os.makedirs(out_folder, exist_ok=True)
                for p in tqdm(file_list, desc=f"Saving {split_name} (.npz)"):
                    try:
                        stem = p.stem
                        hsi = tifffile.imread(p)
                        # ensure C,H,W
                        if hsi.ndim == 3 and hsi.shape[2] == num_hsi_channels:
                            hsi = hsi.transpose(2,0,1)
                        if hsi.shape[0] != num_hsi_channels:
                            print(f"  - skip {p.name}: unexpected channels {hsi.shape}")
                            continue
                        hsi_tensor = torch.from_numpy(hsi.astype(np.float32))
                        # apply same (empty) transform as labelled pipeline
                        hsi_norm = hsi_transform(hsi_tensor)

                        # try to load RGB; if not found, create zeros matching H,W
                        rgb_path_png = unlabeled_rgb_folder / f"{stem}.png"
                        rgb_path_jpg = unlabeled_rgb_folder / f"{stem}.jpg"
                        if rgb_path_png.exists() or rgb_path_jpg.exists():
                            rgb_p = rgb_path_png if rgb_path_png.exists() else rgb_path_jpg
                            rgb_pil = Image.open(str(rgb_p)).convert('RGB')
                            rgb_t = rgb_transform(rgb_pil)  # may be (3,256,256)
                        else:
                            H, W = hsi_norm.shape[1], hsi_norm.shape[2]
                            rgb_t = torch.zeros((3, H, W), dtype=torch.float32)

                        # Ensure RGB spatial dims match HSI spatial dims before saving
                        if rgb_t.shape[1:] != (hsi_norm.shape[1], hsi_norm.shape[2]):
                            import torch.nn.functional as F
                            rgb_t = F.interpolate(rgb_t.unsqueeze(0), size=(hsi_norm.shape[1], hsi_norm.shape[2]),
                                                    mode='bilinear', align_corners=False).squeeze(0)

                        # Save as named arrays for consistency with labeled pipeline
                        np.savez_compressed(out_folder / f"{stem}.npz",
                                            rgb=rgb_t.numpy(),
                                            hsi=hsi_norm.numpy())
                    except Exception as e:
                        print(f"Error processing {p.name}: {e}")

            # run
            print(f"Unlabeled files: {n} -> val: {len(val_files)}, train: {len(train_files)} (seed={random_seed})")
            _process_list(val_files, 'val')
            _process_list(train_files, 'train')
            print("Finished unlabeled -> .npz conversion (normalized).")